# Реальные данные: выбираем модель и проверяем её
### Рабочая тетрадь
*Курс: Байесовский анализ эмпирических данных (2026). Дополнение к занятиям 3 и 4.*

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/03_04_real_data_conjugate_cases_ru.ipynb)

На занятии 3 мы генерировали данные по заданной модели. В этой тетради начнём с реальных данных и рассмотрим, как выбрать модель, оценить её параметры и проверить, насколько хорошо она описывает наблюдения.

Разберём четыре примера, следуя общей последовательности:

| Шаг | Основной вопрос |
|---|---|
| 1. Изучение данных | Какие значения принимает переменная и как они распределены? |
| 2. Выбор модели | Какое распределение подходит для описания этих наблюдений? |
| 3. Выбор априорного распределения | Какие значения параметров правдоподобны и какие данные предсказывает модель до учёта наблюдений? |
| 4. Обновление по данным | Как изменяется распределение параметров после учёта наблюдений? |
| 5. Проверка модели | Воспроизводит ли модель существенные особенности данных? |
| 6. Выводы | Для каких задач модель подходит и в чём состоят её ограничения? |

Примеры показывают разные стороны моделирования: влияние априорных предположений, несоответствие модели данным, выбор шкалы измерения и потерю информации при объединении категорий.

> **Роль сопряжённых распределений**
>
> Во всех четырёх примерах используются сопряжённые пары: сочетания априорного распределения и правдоподобия, для которых апостериорное распределение можно получить аналитически. Это позволяет сосредоточиться на выборе и проверке модели, не обращаясь к MCMC.
>
> Вывод формул сопряжённого обновления на занятии не рассматривается. Формулы приведены для самостоятельного обращения к ним; пояснения и выбор гиперпараметров обсуждаются в приложениях 4A и 4C. Для работы с тетрадью достаточно уметь подставить в формулы параметры и характеристики данных.
>
> Для более сложных моделей аналитическое решение часто недоступно. Численные методы байесовского вывода рассматриваются на занятии 6.

## 1. Перед занятием

Цель занятия — научиться выбирать распределение для реальных социологических данных, обосновывать априорные предположения и оценивать соответствие модели наблюдениям.

- **Предварительная подготовка:** занятие 3 о генеративных моделях и правдоподобии, основные понятия занятия 4 об априорном и апостериорном распределениях.
- **Продолжительность:** около 120 минут. Примеры можно разбирать по отдельности, предварительно выполнив ячейку загрузки данных; пример 4 содержательно продолжает пример 1.
- **Домашнее задание:** анализ симулированных данных о ДТП в условиях топливного кризиса. Постановка и задания приведены в конце тетради.

Далее используются как полные названия «априорное распределение» и «апостериорное распределение», так и их краткие варианты — «приор» и «постериор».

## 2. Создайте рабочую копию

Чтобы сохранять изменения в Google Colab, выберите **Файл → Сохранить копию на Диске** (File → Save a copy in Drive).

## 3. Окружение и загрузка данных

В тетради используются три набора данных из примеров к книге **Regression and Other Stories** (Gelman, Hill, Vehtari). Они подготовлены для курса скриптом `scripts/prepare_real_data_cases.py`.

В Google Colab файлы автоматически загружаются из репозитория курса на GitHub. При локальном запуске используются файлы из папки `data`, если они доступны; в противном случае данные также загружаются с GitHub.

In [ ]:
import sys
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    import plotly.io as pio
    pio.renderers.default = "colab"
    print("⚡ Работаем в Google Colab.")
else:
    print("💻 Работаем в локальном окружении.")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

REPO = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/"


def load(name):
    """Читает файл из репозитория курса; при локальном запуске ищет папку data/ рядом."""
    import os
    path = REPO + name
    if not IS_COLAB:
        for up in ("data", "../data", "../../data"):
            candidate = os.path.join(up, name)
            if os.path.exists(candidate):
                path = candidate
                break
    df = pd.read_csv(path)
    print(f"  {name}: {df.shape[0]} строк, {df.shape[1]} столбцов")
    return df


print("Загружаем данные:")
pew = load("pew_attendance_by_state.csv")
risky = load("risky_behavior.csv")
earnings = load("earnings_1990.csv")
print(f"✅ Готово. Начальное значение генератора: {RANDOM_SEED}.")

## 4. Источники данных и ограничения интерпретации

| Набор | Содержание | Единица наблюдения и формат файла | Объём |
|---|---|---|---|
| `pew_attendance_by_state.csv` | Опрос Pew Research Center о частоте посещения религиозных служб, США, июнь 2008 года | Ответ респондента; в файле представлены суммы по штатам | 25 894 респондента в 50 штатах |
| `risky_behavior.csv` | Исследование профилактики ВИЧ: число незащищённых половых актов до и после вмешательства | Пара | 434 пары |
| `earnings_1990.csv` | Обследование доходов, США, 1990 год | Респондент | 1 816 респондентов |

Все три набора содержат реальные обезличенные данные из [примеров к Regression and Other Stories](https://avehtari.github.io/ROS-Examples/examples.html). Данные Pew представлены только в агрегированном виде, без индивидуальных анкет.

Эти исследования используются для изучения методов моделирования. Их результаты нельзя непосредственно переносить на современную Россию или другие группы населения. В примере о профилактике ВИЧ обсуждайте наблюдаемые характеристики поведения, избегая оценочных суждений об участниках.

Данные о ДТП в домашнем задании **симулированы**. Задание использует контекст топливного кризиса, но числовые значения заданы в учебных целях и не являются статистикой ГИБДД.

## 5. Учебные цели

После работы с тетрадью вы сможете:

1. Предложить распределение для наблюдаемой переменной, учитывая её допустимые значения, форму распределения и условия сбора данных.
2. Проверить априорные предположения с помощью предсказаний на шкале наблюдений — априорного предиктивного распределения (*prior predictive distribution*).
3. Получить апостериорное распределение по формуле сопряжённого обновления и назвать допущения, при которых эта формула применима.
4. Сопоставить данные с апостериорными предсказаниями модели (*posterior predictive check*) и объяснить, почему узкий апостериорный интервал сам по себе не свидетельствует о хорошем соответствии модели данным.
5. Оценить чувствительность выводов к выбору приора при разных объёмах выборки.

## 6. Предварительное обсуждение

До выполнения кода кратко ответьте на вопросы. Позже сравните свои предположения с результатами анализа.

1. В каком диапазоне могут находиться доли жителей штатов США, посещающих религиозные службы не реже раза в неделю? Насколько велики различия между штатами?
2. Как может быть распределено число незащищённых половых актов за определённый период у одной пары? Есть ли основания ожидать равенства среднего и дисперсии, как в распределении Пуассона?
3. Какую форму вы ожидаете увидеть у распределения годового дохода? Какие ограничения могут возникнуть при использовании нормальной модели?
4. От каких особенностей данных и априорных предположений будет зависеть влияние приора на результат?

---
# Пример 1. Beta–Binomial: посещение религиозных служб в штатах США

**Исследовательский вопрос:** какая доля взрослых жителей штата посещает религиозные службы не реже раза в неделю?

## Шаг 1. Изучение данных

Выполните ячейку. Сравните объёмы выборок по штатам и распределение наблюдаемых долей.

In [ ]:
pew["p_hat"] = pew.weekly_or_more / pew.n_respondents
national = pew.weekly_or_more.sum() / pew.n_respondents.sum()

print(f"Всего респондентов: {pew.n_respondents.sum():,}, штатов: {len(pew)}")
print(f"Общая доля в выборке (еженедельно или чаще): {national:.3f}")
print(f"\nРазмер выборки по штатам: от {pew.n_respondents.min()} до {pew.n_respondents.max()}")
print("\nШтаты с наименьшими выборками:")
print(pew.nsmallest(5, "n_respondents")[["state", "n_respondents", "weekly_or_more", "p_hat"]].to_string(index=False))
print("\nШтаты с наибольшими выборками:")
print(pew.nlargest(3, "n_respondents")[["state", "n_respondents", "weekly_or_more", "p_hat"]].to_string(index=False))

big = pew[pew.n_respondents >= 20]          # для гистограммы оставляем штаты с n ≥ 20; случай n=1 рассмотрим отдельно
fig = go.Figure()
fig.add_trace(go.Histogram(x=big.p_hat, nbinsx=20, marker_color="#2b6cb0",
                           hovertemplate="доля %{x}<br>штатов: %{y}<extra></extra>"))
fig.add_vline(x=national, line_dash="dash", line_color="black",
              annotation_text=f"общая выборка {national:.2f}")
fig.update_layout(title="Доля посещающих службы еженедельно: штаты США (Pew, 2008; n ≥ 20)",
                  xaxis_title="доля в штате", yaxis_title="число штатов",
                  template="plotly_white", height=380)
fig.show()

## Шаг 2. Выбор распределения

**Задание.** Определите единицу наблюдения и возможные значения ответа. Чем индивидуальные ответы отличаются от данных, представленных в файле?

<details><summary><b>Пояснение</b></summary>

Единица наблюдения — респондент. После объединения категорий его ответ принимает два значения: посещает службы не реже раза в неделю (1) или реже (0). Такой ответ можно описать распределением Бернулли.

В файле ответы агрегированы по штатам. Если внутри штата они условно независимы при общей вероятности $\theta$, число ответов «не реже раза в неделю» имеет биномиальное распределение:

$$k \mid n, \theta \sim \operatorname{Binomial}(n, \theta).$$

Параметр $\theta$ — вероятность, поэтому его значения лежат в $[0, 1]$. Бета-распределение подходит для задания приора на этой шкале и образует сопряжённую пару с биномиальным правдоподобием.

**Упрощение модели.** Мы не учитываем веса опроса, стратификацию и неответы. Поэтому полученные оценки служат учебной иллюстрацией и не заменяют анализ с учётом дизайна обследования.
</details>

## Шаг 3. Априорные распределения и предсказания

Рассмотрим три приора для $\theta$. Сначала сравним их априорные предсказания при объёме выборки Вайоминга, а затем оценим влияние на результаты для Вайоминга и Калифорнии.

Один из приоров центрирован на общей доле ответов в этом же наборе данных. Это учебный пример эмпирического выбора приора: такая информация не является независимой от анализируемой выборки.

In [ ]:
focus = {}
for st in ["wyoming", "california"]:
    row = pew.loc[pew.state == st].iloc[0]
    focus[st] = (int(row.weekly_or_more), int(row.n_respondents))
    print(f"{st}: k = {int(row.weekly_or_more)}, n = {int(row.n_respondents)}, доля = {row.p_hat:.3f}")

# Три приора на theta
neff = 50                                    # "сила" информативного приора в единицах наблюдений
PRIORS = {
    "равномерный Beta(1,1)":            (1.0, 1.0),
    "симметричный Beta(2,2)":             (2.0, 2.0),
    f"по общей выборке (n_eff={neff})": (national * neff, (1 - national) * neff),
}

theta_grid = np.linspace(0, 1, 500)
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "<b>Приоры на θ</b>",
    "<b>Априорное предсказание: сколько «да» из 28</b><br><span style='font-size:11px'>размер выборки Вайоминга</span>"])

k_wy, n_wy = focus["wyoming"]
for (lab, (a, b)), c in zip(PRIORS.items(), ["#94a3b8", "#2b6cb0", "#c53030"]):
    fig.add_trace(go.Scatter(x=theta_grid, y=stats.beta.pdf(theta_grid, a, b),
                             name=lab, line=dict(color=c, width=2.5)), row=1, col=1)
    th = rng.beta(a, b, 4000)
    k_sim = rng.binomial(n_wy, th)
    fig.add_trace(go.Histogram(x=k_sim, histnorm="probability", marker_color=c,
                               opacity=0.55, name=lab, showlegend=False,
                               xbins=dict(size=1)), row=1, col=2)

fig.add_vline(x=k_wy, line_dash="dash", line_color="black",
              annotation_text=f"наблюдали k={k_wy}", row=1, col=2)
fig.update_layout(template="plotly_white", height=400, barmode="overlay",
                  legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title_text="θ", row=1, col=1)
fig.update_xaxes(title_text="k (из 28)", row=1, col=2)
fig.show()

print("Наблюдаемое k = 9 имеет ненулевую вероятность при всех трёх приорах.")
print("При равномерном Beta(1,1) все значения k от 0 до 28 равновероятны:")
print("в том числе выборки, где ни один или все респонденты посещают службы еженедельно.")

## Шаг 4. Апостериорное распределение

Бета-приор и биномиальное правдоподобие образуют сопряжённую пару:

$$\theta \sim \operatorname{Beta}(a, b), \quad k \mid \theta \sim \operatorname{Binomial}(n, \theta)
\;\Longrightarrow\; \theta \mid k \sim \operatorname{Beta}(a + k,\; b + n - k).$$

При обновлении к параметру $a$ прибавляется число ответов «не реже раза в неделю», а к $b$ — число остальных ответов. Апостериорное распределение при этом остаётся бета-распределением.

Вывод формулы приведён в приложении 4A и на занятии не рассматривается.

## Шаг 5. Сравнение апостериорных оценок

На графиках показаны апостериорные распределения при трёх приорах; пунктирная линия обозначает выборочную долю. Сравните положение и ширину распределений для двух штатов. Это анализ чувствительности оценок к приору; сам по себе он не проверяет все предположения биномиальной модели.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"<b>Вайоминг</b>: k={focus['wyoming'][0]}, n={focus['wyoming'][1]}",
    f"<b>Калифорния</b>: k={focus['california'][0]}, n={focus['california'][1]}"])

summary = []
for col, st in enumerate(["wyoming", "california"], start=1):
    k, n = focus[st]
    for (lab, (a, b)), c in zip(PRIORS.items(), ["#94a3b8", "#2b6cb0", "#c53030"]):
        pa, pb = a + k, b + n - k
        fig.add_trace(go.Scatter(x=theta_grid, y=stats.beta.pdf(theta_grid, pa, pb),
                                 name=lab, line=dict(color=c, width=2.5),
                                 showlegend=(col == 1)), row=1, col=col)
        lo, hi = stats.beta.ppf([0.025, 0.975], pa, pb)
        summary.append({"штат": st, "приор": lab, "постериорное среднее": pa / (pa + pb),
                        "95% интервал": f"[{lo:.3f}, {hi:.3f}]",
                        "ширина": hi - lo, "сдвиг от k/n": pa / (pa + pb) - k / n})
    fig.add_vline(x=k / n, line_dash="dot", line_color="black", row=1, col=col)

fig.update_layout(template="plotly_white", height=400,
                  legend=dict(orientation="h", y=-0.2),
                  title="Постериор θ: три приора, два штата (пунктир — доля в выборке)")
fig.update_xaxes(title_text="θ", range=[0, 0.8])
fig.show()

print(pd.DataFrame(summary).to_string(index=False, float_format=lambda v: f"{v:+.3f}"))

### Оценивание при одном наблюдении: Гавайи

В наборе данных на Гавайи приходится один респондент. Он не вошёл в категорию посещающих службы не реже раза в неделю. Сравним оценку максимального правдоподобия с апостериорными оценками при разных приорах.

In [ ]:
k_hi = int(pew.loc[pew.state == 'hawaii', 'weekly_or_more'].iloc[0])
n_hi = int(pew.loc[pew.state == 'hawaii', 'n_respondents'].iloc[0])
print(f'Гавайи: k = {k_hi}, n = {n_hi}')
print(f'Оценка максимального правдоподобия: {k_hi / n_hi:.3f}')
print('  Нулевая точечная оценка получена по единственному наблюдению.')
print('  Она не означает, что соответствующая доля в населении заведомо равна нулю.')
print()

for lab, (a, b) in PRIORS.items():
    pa, pb = a + k_hi, b + n_hi - k_hi
    lo, hi = stats.beta.ppf([0.025, 0.975], pa, pb)
    print(f'  {lab:30s} постериорное среднее {pa/(pa+pb):.3f}, 95% [{lo:.3f}, {hi:.3f}]')

print()
print('При всех трёх приорах апостериорное среднее больше нуля.')
print('При одном наблюдении оценка и ширина интервала существенно зависят от приора.')
print()
print('Изменение влияния приора при разных объёмах выборки:')
a_nat, b_nat = PRIORS[f'по общей выборке (n_eff={neff})']
for st in ['hawaii', 'wyoming', 'california']:
    row = pew.loc[pew.state == st].iloc[0]
    k, n = int(row.weekly_or_more), int(row.n_respondents)
    shift = (a_nat + k) / (a_nat + b_nat + n) - k / n
    print(f'  {st:12s} n={n:5d}  сдвиг оценки приором: {shift:+.3f}')

## Шаг 6. Выводы по примеру 1

**Задание.** Сравните результаты для Вайоминга и Калифорнии. Насколько апостериорное среднее при информативном приоре отличается от выборочной доли? Объясните различие между штатами.

<details><summary><b>Пояснение</b></summary>

В Вайоминге ($n = 28$) приор, центрированный на общей доле по выборке, сдвигает апостериорное среднее примерно на $+0{,}055$ и сужает интервал. Его концентрация $a+b=50$ велика относительно объёма выборки, поэтому априорные предположения существенно влияют на оценку.

В Калифорнии ($n = 2350$) сдвиг составляет около $+0{,}002$. При выбранных приорах апостериорные распределения близки друг к другу: вклад данных значительно больше вклада априорной информации.

Таким образом, влияние приора определяется соотношением его информативности и информации в данных. При небольшой выборке особенно важно обосновать приор и показать анализ чувствительности. Большой объём выборки уменьшает влияние рассматриваемых здесь приоров, но не делает выбор приора несущественным во всех моделях.

Иерархическая модель позволяет совместно оценивать доли по штатам и степень их различия. Тогда оценки для малых выборок частично смещаются к общему уровню, причём степень такого смещения оценивается по данным всех штатов. Этот подход рассматривается на занятии 10.
</details>

---
### Обсуждение 1

В первом примере параметр — вероятность на отрезке $[0, 1]$. В следующем примере параметр — положительное среднее число событий без заданной верхней границы. Какие распределения можно использовать для приора на такой параметр?

---
# Пример 2. Gamma–Poisson: проверка модели для счётных данных

**Исследовательский вопрос:** сколько незащищённых половых актов в среднем приходится на пару после профилактического вмешательства?

В исследовании профилактики ВИЧ участвовали 434 пары. Для каждой пары зарегистрировано число незащищённых половых актов до (`acts_before`) и после (`acts_after`) вмешательства. Здесь мы моделируем распределение числа событий после вмешательства; оценка его эффекта потребовала бы отдельного анализа парных наблюдений.

В домашнем задании также будут рассматриваться счётные данные за два периода, но единицей наблюдения там станет день.

## Шаг 1. Изучение данных

In [ ]:
y = risky.acts_after.values.astype(int)
y_before = risky.acts_before.values.astype(int)

for lab, v in [("до интервенции", y_before), ("после интервенции", y)]:
    print(f"{lab:18s}: n={len(v)}, среднее={v.mean():6.2f}, дисперсия={v.var(ddof=1):8.2f}, "
          f"дисперсия/среднее={v.var(ddof=1)/v.mean():6.2f}, нулей={(v==0).mean():5.1%}, максимум={v.max()}")

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "<b>Наблюдаемые данные (после)</b>", "<b>До и после</b>"])
fig.add_trace(go.Histogram(x=y, xbins=dict(size=5), marker_color="#c53030", name="после"), row=1, col=1)
fig.add_trace(go.Histogram(x=y_before, xbins=dict(size=5), histnorm="probability",
                           marker_color="#2b6cb0", opacity=0.6, name="до"), row=1, col=2)
fig.add_trace(go.Histogram(x=y, xbins=dict(size=5), histnorm="probability",
                           marker_color="#c53030", opacity=0.6, name="после"), row=1, col=2)
fig.update_layout(template="plotly_white", height=380, barmode="overlay")
fig.update_xaxes(title_text="число актов за период", range=[0, 120])
fig.update_yaxes(title_text="число пар", row=1, col=1)
fig.show()

## Шаг 2. Выбор распределения

**Задание.** Переменная принимает неотрицательные целые значения и не имеет заданной верхней границы. Подходит ли для неё распределение Пуассона? Какое ограничение эта модель накладывает на соотношение среднего и дисперсии?

<details><summary><b>Пояснение</b></summary>

В качестве исходной модели рассмотрим $y_i \mid \lambda \sim \operatorname{Poisson}(\lambda)$. В распределении Пуассона математическое ожидание и дисперсия равны $\lambda$.

В данных отношение выборочной дисперсии к среднему составляет около 44. Такое превышение дисперсии над средним называется **сверхдисперсией** и указывает на возможное несоответствие пуассоновской модели. Тем не менее проведём оценивание, чтобы увидеть, как это несоответствие проявляется в апостериорной предиктивной проверке.

Для $\lambda > 0$ выберем гамма-приор: $\lambda \sim \operatorname{Gamma}(a, b)$. Здесь $a$ — параметр формы, $b$ — параметр интенсивности (*rate*), а среднее равно $a/b$. В функциях SciPy и NumPy ниже используется параметр масштаба `scale = 1/b`.
</details>

## Шаг 3. Априорные распределения и предсказания

In [ ]:
GAMMA_PRIORS = {
    "широкий Gamma(2, 0.1)":       (2.0, 0.1),     # среднее 20, очень широкий
    "информативный Gamma(20, 1)": (20.0, 1.0),    # среднее 20, меньший разброс
    "скептический Gamma(50, 5)":  (50.0, 5.0),    # среднее 10, узкий
}

lam_grid = np.linspace(0.1, 60, 400)
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "<b>Приоры на λ</b>", "<b>Априорное предсказание: одно наблюдение</b>"])
for (lab, (a, b)), c in zip(GAMMA_PRIORS.items(), ["#94a3b8", "#2b6cb0", "#c53030"]):
    fig.add_trace(go.Scatter(x=lam_grid, y=stats.gamma.pdf(lam_grid, a, scale=1/b),
                             name=lab, line=dict(color=c, width=2.5)), row=1, col=1)
    lam_draw = rng.gamma(a, 1/b, 4000)
    fig.add_trace(go.Histogram(x=rng.poisson(lam_draw), histnorm="probability", opacity=0.55,
                               marker_color=c, name=lab, showlegend=False,
                               xbins=dict(size=2)), row=1, col=2)
fig.add_vline(x=y.mean(), line_dash="dash", line_color="black",
              annotation_text=f"среднее в данных {y.mean():.1f}", row=1, col=2)
fig.update_layout(template="plotly_white", height=400, barmode="overlay",
                  legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title_text="λ", row=1, col=1)
fig.update_xaxes(title_text="y (одно наблюдение)", range=[0, 70], row=1, col=2)
fig.show()

print("Сопоставьте правый хвост априорных предсказаний с наблюдаемыми значениями:")
print(f"максимум в данных равен {y.max()}. Соответствие хвоста проверим после обновления.")

## Шаг 4. Апостериорное распределение

Гамма-приор сопряжён пуассоновскому правдоподобию:

$$\lambda \sim \operatorname{Gamma}(a, b), \quad y_i \mid \lambda \sim \operatorname{Poisson}(\lambda)
\;\Longrightarrow\; \lambda \mid y \sim \operatorname{Gamma}\!\left(a + \sum_i y_i,\; b + n\right).$$

К параметру формы прибавляется суммарное число событий, а к параметру интенсивности — число наблюдений. Формула даёт апостериорное распределение в рамках выбранной модели, но не показывает, насколько обоснованы её предположения. Для этого нужна отдельная проверка.

In [ ]:
rows = []
for lab, (a, b) in GAMMA_PRIORS.items():
    pa, pb = a + y.sum(), b + len(y)
    lo, hi = stats.gamma.ppf([0.025, 0.975], pa, scale=1/pb)
    rows.append({"приор": lab, "постериорное среднее λ": pa/pb,
                 "95% интервал": f"[{lo:.2f}, {hi:.2f}]", "ширина": hi - lo})
print(f"Сумма наблюдений = {y.sum()}, n = {len(y)}, выборочное среднее = {y.mean():.3f}\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nПри всех трёх приорах оценки близки, а ширина 95%-го интервала меньше 0.8.")
print("  Эти интервалы рассчитаны при предположении о пуассоновском распределении.")

## Шаг 5. Апостериорная предиктивная проверка

> **Зачем генерировать выборки, если распределение известно?**
>
> В следующей ячейке `rng.gamma(...)` генерирует независимые значения из уже известного апостериорного распределения. Затем для каждого значения $\lambda$ создаётся повторная выборка наблюдений. Так мы получаем данные, которые модель могла бы предсказать с учётом неопределённости параметра.
>
> Здесь симуляция используется для расчёта предсказаний. Квантили самого гамма-распределения можно вычислить непосредственно, например через `stats.gamma.ppf`. В более сложных моделях MCMC позволяет приближённо исследовать апостериорное распределение, для которого такого аналитического решения нет.

Проверим, соответствует ли узкому апостериорному интервалу хорошее описание данных. Сгенерируем повторные выборки по 434 наблюдения и сравним их с исходными данными по стандартному отклонению, максимуму и доле нулей. Эти характеристики позволяют оценить особенности распределения, которые не описываются одним средним.

In [ ]:
a_post, b_post = 2.0 + y.sum(), 0.1 + len(y)
lam_post = rng.gamma(a_post, 1/b_post, 3000)
y_rep = rng.poisson(lam_post[:, None], size=(3000, len(y)))

STATS = {
    "стандартное отклонение": lambda v: v.std(ddof=1),
    "максимум":               lambda v: v.max(),
    "доля нулей, %":          lambda v: (v == 0).mean() * 100,
}

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"<b>{k}</b>" for k in STATS])
print(f"{'статистика':24s} {'в данных':>12s}   предсказание модели (95%)   результат")
print("-" * 82)
for i, (lab, fn) in enumerate(STATS.items(), start=1):
    obs = fn(y)
    sim = np.array([fn(r) for r in y_rep])
    lo, hi = np.percentile(sim, [2.5, 97.5])
    ok = "ok" if lo <= obs <= hi else "вне интервала"
    print(f"{lab:24s} {obs:12.2f}   [{lo:8.2f}, {hi:8.2f}]      {ok}")
    fig.add_trace(go.Histogram(x=sim, marker_color="#2b6cb0", showlegend=False), row=1, col=i)
    fig.add_vline(x=obs, line_color="#c53030", line_width=3, row=1, col=i)

fig.update_layout(template="plotly_white", height=330,
                  title="Предиктивная проверка: синим — статистики повторных выборок, красным — наблюдаемое значение")
fig.show()

In [ ]:
# Сравнение данных с пуассоновской и отрицательной биномиальной моделями
xs = np.arange(0, 121)
nb_size = y.mean()**2 / (y.var(ddof=1) - y.mean())
nb_prob = nb_size / (nb_size + y.mean())

fig = go.Figure()
fig.add_trace(go.Histogram(x=y, xbins=dict(size=5), histnorm="probability density",
                           marker_color="#c53030", opacity=0.6, name="реальные данные"))
fig.add_trace(go.Scatter(x=xs, y=stats.poisson.pmf(xs, y.mean()), name=f"Пуассон(λ={y.mean():.1f})",
                         line=dict(color="#2b6cb0", width=3)))
fig.add_trace(go.Scatter(x=xs, y=stats.nbinom.pmf(xs, nb_size, nb_prob),
                         name=f"Отрицательное биномиальное", line=dict(color="#276749", width=3)))
fig.update_layout(template="plotly_white", height=400, xaxis_range=[0, 120],
                  title="Одни и те же данные, две модели",
                  xaxis_title="число актов", yaxis_title="плотность")
fig.show()

print(f"Отрицательное биномиальное = смесь Гамма-Пуассон: у каждой пары свой λ_i ~ Gamma.")
print(f"Подобранный параметр формы: {nb_size:.3f} (чем меньше, тем сильнее разброс между парами).")

## Шаг 6. Выводы по примеру 2

**Задание.** Апостериорный 95%-й интервал для $\lambda$ имеет ширину меньше единицы, однако наблюдаемые значения трёх проверочных статистик не согласуются с предсказаниями. Объясните, почему эти результаты не противоречат друг другу.

<details><summary><b>Пояснение</b></summary>

Апостериорное среднее $\lambda \approx 16{,}5$ близко к выборочному среднему. При этом модель существенно недооценивает вариативность наблюдений. В пуассоновском распределении со средним 16,5 стандартное отклонение составляет около 4, тогда как в данных оно равно 26,8.

Это несоответствие проявляется и в других характеристиках: модель предсказывает максимумы примерно от 27 до 34 вместо наблюдаемого значения 200 и почти не предсказывает нулей, составляющих около 29% выборки.

**Узкий апостериорный интервал характеризует неопределённость при условии принятой модели.** Если модель неверно описывает разброс данных, такой интервал может создавать преувеличенное представление о точности оценки среднего. Близость оценки к выборочному среднему не гарантирует надёжности интервала.

Одно из возможных объяснений сверхдисперсии — различия между парами. Если допустить индивидуальные интенсивности $\lambda_i$ с гамма-распределением, маргинальное распределение числа событий будет отрицательным биномиальным. На графике оно лучше отражает большой разброс наблюдений. Это перспективное направление уточнения модели, которое также требует проверки, особенно по доле нулей. Иерархические модели рассматриваются на занятии 10.

В домашнем задании следует аналогичным образом проверить предсказания, прежде чем интерпретировать оценку изменения числа ДТП.
</details>

---
### Обсуждение 2

В примере 1 оценки для Вайоминга заметно зависели от приора. В примере 2 оценки почти не зависели от приора, но модель плохо описывала данные. Чем различаются эти проблемы и как обнаружить каждую из них?

---
# Пример 3. Normal–Normal: моделирование доходов на логарифмической шкале

**Исследовательский вопрос:** каков типичный годовой доход участников обследования 1990 года?

В качестве характеристики типичного дохода рассмотрим медиану. Сначала построим модель для положительных доходов, а затем обсудим, как наличие нулевых значений ограничивает её интерпретацию.

## Шаг 1. Изучение данных

In [ ]:
earn_all = earnings.earn.dropna().values
pos = earn_all[earn_all > 0]
ly = np.log(pos)

print(f"Всего респондентов: {len(earn_all)}, из них с нулевым доходом: {(earn_all == 0).sum()} ({(earn_all == 0).mean():.1%})")
print(f"Доходы > 0: n={len(pos)}, среднее={pos.mean():,.0f}, медиана={np.median(pos):,.0f}, "
      f"максимум={pos.max():,.0f}, скошенность={pd.Series(pos).skew():.2f}")
print(f"log(доход):  среднее={ly.mean():.4f}, ст.откл.={ly.std(ddof=1):.4f}, скошенность={pd.Series(ly).skew():.2f}")

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "<b>Исходная шкала</b><br><span style='font-size:11px'>скошенность ≈ 5</span>",
    "<b>Логарифмическая шкала</b><br><span style='font-size:11px'>скошенность ≈ −0.8</span>"])
fig.add_trace(go.Histogram(x=pos, nbinsx=60, marker_color="#94a3b8"), row=1, col=1)
fig.add_trace(go.Histogram(x=ly, nbinsx=45, marker_color="#d97706"), row=1, col=2)
fig.update_layout(template="plotly_white", height=370, showlegend=False)
fig.update_xaxes(title_text="годовой доход, $", row=1, col=1)
fig.update_xaxes(title_text="log(годовой доход)", row=1, col=2)
fig.show()

## Шаг 2. Выбор распределения и шкалы

**Задание.** Какие особенности доходов затрудняют применение нормальной модели на исходной шкале? Как меняется распределение после логарифмирования?

<details><summary><b>Пояснение</b></summary>

Доходы в этом наборе неотрицательны и имеют выраженную правую асимметрию: коэффициент асимметрии положительных доходов составляет около 5, максимум — 400 000 долларов при медиане около 19 000. Нормальная модель симметрична и допускает отрицательные значения, поэтому плохо соответствует этим данным на исходной шкале.

После логарифмирования распределение становится ближе к симметричному, хотя коэффициент асимметрии остаётся около $-0{,}8$. Это даёт основание рассмотреть нормальную модель для логарифма положительного дохода, то есть логнормальную модель на исходной шкале:

$$\log(\text{доход}) \sim \operatorname{Normal}(\mu, \sigma).$$

Такое описание совместимо с предположением о мультипликативных различиях в доходах: например, о прибавках в процентах. Однако форма распределения сама по себе не устанавливает механизм формирования дохода.

**Упрощение модели.** Мы фиксируем $\sigma$ на уровне выборочного стандартного отклонения логарифмов и не учитываем неопределённость этой оценки. При таком условии нормальный приор для $\mu$ даёт простую формулу обновления. Совместное оценивание среднего и дисперсии с нормально-обратно-гамма-приором рассматривается в приложении 4C.

Нулевые доходы, составляющие 10,3% выборки, нельзя логарифмировать. Пока мы исключаем их из анализа; следовательно, выводы относятся только к участникам с положительным доходом. К этому ограничению вернёмся на шаге 6.
</details>

## Шаг 3. Интерпретация приоров на шкале дохода

Параметр $\mu$ задан на логарифмической шкале. Для содержательной интерпретации рассмотрим соответствующий медианный доход $\exp(\mu)$ в долларах. В таблице ниже приведены априорные интервалы для этой медианы, а на графике — априорные предсказания дохода отдельного человека. Это разные величины: предсказания учитывают также индивидуальный разброс доходов.

In [ ]:
sigma = ly.std(ddof=1)     # считаем известной
NORMAL_PRIORS = {
    "широкий: N(log 20000, 1.0)":      (np.log(20000), 1.0),
    "информативный: N(log 50000, 0.2)":   (np.log(50000), 0.2),
    "очень широкий: N(log 20000, 3.0)": (np.log(20000), 3.0),
}

print(f"σ (принята известной) = {sigma:.4f}\n")
print(f"{'приор':34s} {'медиана, $':>14s}   90% интервал приора для медианы, $")
print("-" * 86)
fig = go.Figure()
for (lab, (m0, t0)), c in zip(NORMAL_PRIORS.items(), ["#2b6cb0", "#c53030", "#94a3b8"]):
    lo, hi = np.exp(stats.norm.ppf([0.05, 0.95], m0, t0))
    print(f"{lab:34s} {np.exp(m0):14,.0f}   [{lo:>10,.0f}, {hi:>12,.0f}]")
    draws = rng.normal(m0, t0, 4000)
    y_prior_pred = np.exp(rng.normal(draws, sigma))     # prior predictive для ОДНОГО человека
    fig.add_trace(go.Histogram(x=y_prior_pred[y_prior_pred < 300000], nbinsx=60,
                               histnorm="probability", opacity=0.55,
                               marker_color=c, name=lab))
fig.add_vline(x=np.median(pos), line_dash="dash", line_color="black",
              annotation_text=f"медиана в данных ${np.median(pos):,.0f}")
fig.update_layout(template="plotly_white", height=400, barmode="overlay",
                  title="Априорное предсказание: доход одного человека, доллары",
                  xaxis_title="годовой доход, $", legend=dict(orientation="h", y=-0.25))
fig.show()

print("\nПриор с наибольшим разбросом допускает медианный доход в миллионы долларов.")
print("Приор с меньшим разбросом сосредоточен около медианного дохода $50 000.")
print("Сравним, как эти предположения влияют на апостериорные оценки.")

## Шаг 4. Апостериорное распределение

При фиксированном $\sigma$ нормальный приор сопряжён нормальному правдоподобию. Поэтому апостериорное распределение для $\mu$ также нормально:

$$\mu \sim \operatorname{Normal}(\mu_0, \tau_0), \quad y_i \mid \mu \sim \operatorname{Normal}(\mu, \sigma)
\;\Longrightarrow\; \mu \mid y \sim \operatorname{Normal}(\mu_n, \tau_n),$$

$$\tau_n^{2} = \left(\frac{1}{\tau_0^{2}} + \frac{n}{\sigma^{2}}\right)^{-1}, \qquad
\mu_n = \tau_n^{2}\left(\frac{\mu_0}{\tau_0^{2}} + \frac{n\bar y}{\sigma^{2}}\right).$$

Здесь $y_i$ — логарифм положительного дохода, а вторые параметры нормальных распределений обозначают стандартные отклонения. Вывод формул приведён в приложении 4C и на занятии не рассматривается.

Апостериорное среднее — взвешенное среднее априорного ожидания и выборочного среднего. Вес приора равен

$$w_{\text{prior}} = \frac{1/\tau_0^2}{1/\tau_0^2+n/\sigma^2}.$$

При фиксированных $\tau_0$ и $\sigma$ этот вес уменьшается с ростом $n$. В последнем столбце таблицы ниже показаны его значения для выбранных приоров. Сопоставьте результат с примером 1.

In [ ]:
n = len(ly)
print(f"n = {n}, среднее log(дохода) в данных = {ly.mean():.4f} (${np.exp(ly.mean()):,.0f})\n")
print(f"{'приор':34s} {'μ постериор':>12s} {'медиана, $':>12s}   вклад приора")
print("-" * 86)
for lab, (m0, t0) in NORMAL_PRIORS.items():
    post_var = 1 / (1/t0**2 + n/sigma**2)
    post_mu = post_var * (m0/t0**2 + ly.sum()/sigma**2)
    w_prior = (1/t0**2) / (1/t0**2 + n/sigma**2)        # вес приора
    print(f"{lab:34s} {post_mu:12.4f} {np.exp(post_mu):12,.0f}   {w_prior:.5%}")

print(f"\nПри n = {n} вклад каждого приора в оценку среднего указан в таблице выше.")
print("Сравните с Вайомингом из примера 1: n = 28, концентрация информативного приора — 50.")

## Шаг 5. Апостериорная предиктивная проверка

Сравним распределение логарифмов дохода с предсказаниями модели. В качестве проверочной статистики используем коэффициент асимметрии: он позволяет оценить, насколько данные отличаются от симметричного нормального распределения.

In [ ]:
m0, t0 = NORMAL_PRIORS["широкий: N(log 20000, 1.0)"]
post_var = 1 / (1/t0**2 + n/sigma**2)
post_mu = post_var * (m0/t0**2 + ly.sum()/sigma**2)

mu_draws = rng.normal(post_mu, np.sqrt(post_var), 3000)
y_rep_log = rng.normal(mu_draws[:, None], sigma, size=(3000, n))

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "<b>Данные и распределение при апостериорном среднем μ</b>",
    "<b>Проверка: скошенность</b>"])
fig.add_trace(go.Histogram(x=ly, nbinsx=45, histnorm="probability density",
                           marker_color="#d97706", opacity=0.7, name="реальные данные"), row=1, col=1)
xs = np.linspace(ly.min(), ly.max(), 300)
fig.add_trace(go.Scatter(x=xs, y=stats.norm.pdf(xs, post_mu, sigma), name="модель",
                         line=dict(color="#2b6cb0", width=3)), row=1, col=1)

sim_skew = np.array([pd.Series(r).skew() for r in y_rep_log[:800]])
fig.add_trace(go.Histogram(x=sim_skew, marker_color="#2b6cb0", showlegend=False), row=1, col=2)
fig.add_vline(x=pd.Series(ly).skew(), line_color="#c53030", line_width=3, row=1, col=2)
fig.update_layout(template="plotly_white", height=380, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title_text="log(доход)", row=1, col=1)
fig.update_xaxes(title_text="скошенность реплицированной выборки", row=1, col=2)
fig.show()

lo_s, hi_s = np.percentile(sim_skew, [2.5, 97.5])
obs_s = pd.Series(ly).skew()
print(f"скошенность: в данных {obs_s:.3f}, модель предсказывает [{lo_s:.3f}, {hi_s:.3f}] "
      f"-> {'ok' if lo_s <= obs_s <= hi_s else 'вне интервала'}")
print(f"\nАпостериорная оценка медианного дохода: ${np.exp(post_mu):,.0f}")
print(f"95% интервал: [${np.exp(post_mu - 1.96*np.sqrt(post_var)):,.0f}, "
      f"${np.exp(post_mu + 1.96*np.sqrt(post_var)):,.0f}]")

In [ ]:
# Дополнительный анализ: отдельные модели доходов мужчин и женщин
sub = earnings[(earnings.earn > 0)].dropna(subset=["earn", "male"])
res = {}
for male, lab in [(1, "мужчины"), (0, "женщины")]:
    v = np.log(sub.loc[sub.male == male, "earn"].values)
    pv = 1 / (1/1.0**2 + len(v)/sigma**2)
    pm = pv * (np.log(20000)/1.0**2 + v.sum()/sigma**2)
    res[lab] = rng.normal(pm, np.sqrt(pv), 4000)
    print(f"{lab:10s}: n={len(v):5d}, оценка медианного дохода ${np.exp(pm):>8,.0f}")

ratio = np.exp(res["мужчины"] - res["женщины"])
print(f"\nОтношение медианных доходов (мужчины / женщины): {ratio.mean():.3f}, "
      f"95% интервал [{np.percentile(ratio, 2.5):.3f}, {np.percentile(ratio, 97.5):.3f}]")
print("Это описательное сравнение групп в обследовании 1990 года:")
print("оно не учитывает возможные различия в отрасли, стаже и занятости.")

## Шаг 6. Выводы по примеру 3

<details><summary><b>Пояснение</b></summary>

После логарифмирования нормальная модель лучше передаёт общую форму распределения. Однако предиктивная проверка выявляет несоответствие по асимметрии: среди логарифмов дохода сохраняется левый хвост, связанный с очень низкими положительными доходами. Симметричная модель не воспроизводит эту особенность.

Кроме того, анализ не охватывает 10,3% участников с нулевым доходом. Поэтому оценённую медиану нельзя интерпретировать как медиану дохода всей обследованной группы. Нулевой доход также не следует автоматически отождествлять с безработицей: для этого нужны сведения о занятости.

Для описания всей выборки можно рассмотреть **двухчастную модель**: отдельно моделировать вероятность положительного дохода, а затем распределение дохода при условии, что он положителен. При этом выбор распределения для положительной части по-прежнему требует проверки. Необходимость MCMC зависит от конкретной структуры такой модели; само разделение на две части не исключает аналитического обновления.

Выбор преобразования переменной определяет свойства модели и смысл её параметров. Поэтому следует явно указывать, на какой шкале проводится анализ, какие наблюдения он охватывает и какие упрощения используются.
</details>

---
# Пример 4. Dirichlet–Multinomial: анализ всех категорий ответа

В примере 1 мы объединили ответы в две категории: посещает службы не реже раза в неделю или реже. В исходном опросе было шесть категорий. Такое объединение упрощает анализ, но скрывает различия между людьми, которые посещают службы изредка, и теми, кто не посещает их никогда.

Теперь рассмотрим все шесть категорий. Биномиальную модель заменим мультиномиальной, а бета-приор — распределением Дирихле.

## Шаг 1. Изучение данных

In [ ]:
CATS = ["more_than_weekly", "weekly", "monthly", "few_times_year", "seldom", "never"]
LABELS = ["чаще раза<br>в неделю", "раз в неделю", "1-2 раза<br>в месяц",
          "неск. раз<br>в год", "редко", "никогда"]

nat_counts = pew[CATS].sum().values
nat_props = nat_counts / nat_counts.sum()

fig = go.Figure()
fig.add_trace(go.Bar(x=LABELS, y=nat_props, marker_color="#2b6cb0", name="общая выборка"))
for st, c in [("wyoming", "#c53030"), ("mississippi", "#276749")]:
    row = pew.loc[pew.state == st, CATS].values[0]
    fig.add_trace(go.Scatter(x=LABELS, y=row / row.sum(), mode="markers+lines",
                             name=f"{st} (n={row.sum()})", marker=dict(size=10), line=dict(color=c)))
fig.update_layout(template="plotly_white", height=400,
                  title="Частота посещения религиозных служб: шесть категорий",
                  yaxis_title="доля респондентов", legend=dict(orientation="h", y=-0.25))
fig.show()

print("Общее распределение ответов по категориям:")
for lab, cnt, pr in zip(CATS, nat_counts, nat_props):
    print(f"  {lab:20s} {cnt:6d}  {pr:.4f}")

## Шаг 2. Выбор распределения

Ответ каждого респондента относится ровно к одной из $K = 6$ категорий. При условной независимости ответов и общих для штата вероятностях категорий вектор их численностей имеет мультиномиальное распределение:

$$(k_1, \dots, k_K) \mid n, \boldsymbol{\theta} \sim \operatorname{Multinomial}(n, \boldsymbol{\theta}),
\qquad \sum_{j} \theta_j = 1.$$

Параметр $\boldsymbol{\theta}$ — вектор неотрицательных вероятностей, сумма которых равна единице. Множество таких векторов называется симплексом. Для приора на этом множестве удобно использовать **распределение Дирихле** $\operatorname{Dirichlet}(\alpha_1, \dots, \alpha_K)$. При двух категориях распределение вероятности первой категории совпадает с бета-распределением.

## Шаги 3–4. Априорное и апостериорное распределения

Сравним равномерный приор на симплексе и приор, центрированный на общих долях категорий в выборке. Как и в примере 1, второй вариант использует информацию из тех же данных.

Для сопряжённого обновления к каждому параметру приора прибавляется число ответов в соответствующей категории:

$$\boldsymbol{\theta} \mid \mathbf{k} \sim \operatorname{Dirichlet}(\alpha_1 + k_1, \dots, \alpha_K + k_K).$$

На графике показаны апостериорные средние вероятностей и отдельные 95%-е интервалы для каждой категории.

In [ ]:
st = "wyoming"
k_vec = pew.loc[pew.state == st, CATS].values[0].astype(float)
n_st = k_vec.sum()
print(f"{st}: n = {int(n_st)}, числа ответов по категориям = {k_vec.astype(int)}\n")

DIR_PRIORS = {
    "равномерный α=1":                 np.ones(6),
    "по общей выборке, n_eff=50":      nat_props * 50,
}

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"<b>{p}</b>" for p in DIR_PRIORS])
for col, (lab, alpha) in enumerate(DIR_PRIORS.items(), start=1):
    post_alpha = alpha + k_vec
    draws = rng.dirichlet(post_alpha, 4000)
    mean = post_alpha / post_alpha.sum()
    lo = np.percentile(draws, 2.5, axis=0)
    hi = np.percentile(draws, 97.5, axis=0)
    fig.add_trace(go.Bar(x=LABELS, y=mean, marker_color="#2b6cb0", showlegend=False,
                         error_y=dict(type="data", symmetric=False,
                                      array=hi - mean, arrayminus=mean - lo)), row=1, col=col)
    fig.add_trace(go.Scatter(x=LABELS, y=nat_props, mode="markers", name="общая выборка",
                             marker=dict(color="#c53030", size=9, symbol="diamond"),
                             showlegend=(col == 1)), row=1, col=col)

fig.update_layout(template="plotly_white", height=400,
                  title=f"Постериор по категориям, {st} (n={int(n_st)}); ромбы — доли в общей выборке",
                  legend=dict(orientation="h", y=-0.25))
fig.update_yaxes(title_text="θ_j", row=1, col=1)
fig.show()

post = DIR_PRIORS["по общей выборке, n_eff=50"] + k_vec
dr = rng.dirichlet(post, 4000)
print("Постериор (приор по общей выборке), Вайоминг:")
for j, lab in enumerate(CATS):
    print(f"  {lab:20s} среднее {post[j]/post.sum():.3f}  95% "
          f"[{np.percentile(dr[:, j], 2.5):.3f}, {np.percentile(dr[:, j], 97.5):.3f}]")

In [ ]:
# Потеря информации при объединении категорий: два штата с близкой долей "еженедельно",
# но разным распределением остальных категорий
pew["p_never"] = pew.never / pew.n_respondents
d = pew[pew.n_respondents >= 150]

# Ищем пару штатов с почти одинаковой долей «еженедельно», но максимально разной долей «никогда»
best_gap, a_st, b_st = 0.0, None, None
for i in range(len(d)):
    for j in range(i + 1, len(d)):
        ra, rb = d.iloc[i], d.iloc[j]
        if abs(ra.p_hat - rb.p_hat) < 0.02 and abs(ra.p_never - rb.p_never) > best_gap:
            best_gap, a_st, b_st = abs(ra.p_never - rb.p_never), ra.state, rb.state

cand = pew.loc[pew.state.isin([a_st, b_st])].sort_values("p_never")
print("Штаты с близкими долями посещающих службы не реже раза в неделю:\n")
print(cand[["state", "n_respondents", "p_hat", "p_never"]].to_string(index=False,
      float_format=lambda v: f"{v:.3f}"))

print(f"Доля «еженедельно» различается на {abs(cand.p_hat.iloc[0] - cand.p_hat.iloc[1]):.3f}, "
      f"а доля «никогда» — на {best_gap:.3f}.")

if True:
    fig = go.Figure()
    for s, c in [(a_st, "#2b6cb0"), (b_st, "#c53030")]:
        row = pew.loc[pew.state == s, CATS].values[0]
        fig.add_trace(go.Bar(x=LABELS, y=row / row.sum(), name=s, marker_color=c, opacity=0.75))
    fig.update_layout(template="plotly_white", height=380, barmode="group",
                      title=f"{a_st} и {b_st}: различия в распределении шести категорий",
                      yaxis_title="доля", legend=dict(orientation="h", y=-0.25))
    fig.show()

## Шаги 5–6. Сравнение распределений и выводы по примеру 4

**Задание.** Рассмотрите последний график. Какие различия между штатами не отражает показатель «доля посещающих службы не реже раза в неделю»?

<details><summary><b>Пояснение</b></summary>

Штаты с близкой долей еженедельных посещений могут существенно различаться по остальным категориям. Например, в одном штате среди остальных ответов чаще встречаются редкие посещения, а в другом — полное отсутствие посещений. Одного бинарного показателя недостаточно, чтобы описать эти различия.

Мультиномиальная модель сохраняет информацию обо всех категориях. Правило сопряжённого обновления обобщает правило из примера 1: к параметрам приора прибавляются наблюдаемые численности. Однако удобство вычислений само по себе не обосновывает выбор модели.

**Ограничение.** В этой модели порядок категорий не используется: связь между соседними уровнями частоты посещений никак не выделена. Для учёта порядка можно рассмотреть порядковую логистическую регрессию, обсуждаемую на занятии 9.

В этом примере мы сравнили категориальные распределения и влияние приора. Полноценная апостериорная предиктивная проверка здесь не проводилась, поэтому вывод о пригодности модели остаётся ограниченным.
</details>

---
### Итоги четырёх примеров

| Пример | Данные | Правдоподобие | Приор | Основной результат |
|---|---|---|---|---|
| 1 | Число посещающих службы не реже раза в неделю в каждом штате | Binomial | Beta | Влияние выбранных приоров заметно при $n = 28$ и мало при $n = 2350$ |
| 2 | Число событий у 434 пар | Poisson | Gamma | Модель недооценивает разброс и долю нулей, несмотря на узкий апостериорный интервал |
| 3 | Положительные доходы 1 629 участников из 1 816 | Normal для логарифма дохода | Normal | Логарифмирование улучшает описание формы, но остаются асимметрия и проблема нулевых доходов |
| 4 | Частота посещения служб: шесть категорий | Multinomial | Dirichlet | Модель сохраняет различия между категориями, но не использует их порядок |

**Вопросы для письменного ответа**

1. На какие свойства переменной и условия сбора данных следует опираться при выборе распределения?
2. В каком примере узкий апостериорный интервал мог создать ошибочное представление о качестве модели? Как это обнаружила проверка?
3. Как соотносятся объём выборки и информативность приора в примерах 1 и 3? Приведите результаты расчётов.
4. Приведите пример счётной переменной из вашей предметной области. Какие причины могут привести к сверхдисперсии в этих данных?

---
## Домашнее задание: топливный кризис и ДТП

В предыдущих примерах мы начинали с наблюдаемых данных. В этом задании сначала сформулируем модель возможного изменения аварийности, а затем проанализируем симулированные данные.

**Исследовательский вопрос:** как могло измениться среднее число ДТП в регионе во время топливного кризиса 2025 года?

Рассмотрим учебную ситуацию: на фоне дефицита бензина в августе–сентябре региональная комиссия по безопасности дорожного движения хочет оценить изменение числа ДТП. Необходимо различать изменение интенсивности движения и изменение риска аварии при том же объёме движения.

### Постановка задачи

Единица наблюдения — один день в одном регионе. Измеряемая переменная — число зарегистрированных ДТП за день. Она принимает неотрицательные целые значения; фиксированное число испытаний, как в биномиальной модели, здесь не задано. В качестве исходного приближения рассмотрим распределение Пуассона.

Возможные механизмы изменения аварийности действуют в разных направлениях:

| Предполагаемый механизм | Характеристика | Возможное следствие |
|---|---|---|
| Сокращение числа поездок | Объём движения, или экспозиция $e$ | Снижение числа ДТП |
| Очереди на проезжей части, остановки из-за отсутствия топлива, поиск работающей АЗС | Риск на единицу движения $r$ | Увеличение числа ДТП |
| Использование некачественного топлива, стресс водителей | Риск на единицу движения $r$ | Увеличение числа ДТП |

Зададим модель:

$$y_t \mid \lambda_t \sim \operatorname{Poisson}(\lambda_t), \qquad
\lambda_{\text{during}} = \lambda_{\text{before}} \cdot e \cdot r.$$

Здесь $e$ и $r$ обозначают отношения объёма движения и риска во время кризиса к их значениям до кризиса. Их произведение $\rho = e\,r$ — отношение ожидаемого числа ДТП в двух периодах: при $\rho < 1$ оно снижается, при $\rho > 1$ — увеличивается.

> **Ограничение идентифицируемости**
>
> При $e = 0{,}8$, $r = 1{,}1$ и при $e = 0{,}88$, $r = 1{,}0$ произведение одинаково: $\rho = 0{,}88$. При одной и той же исходной интенсивности оба сочетания дают одинаковое распределение числа ДТП. Поэтому по одним дневным числам ДТП можно оценить $\rho$, но нельзя определить $e$ и $r$ по отдельности. Для их разделения нужны дополнительные сведения об объёме движения, например данные транспортных счётчиков или показатели мобильности.

### Данные

Следующая ячейка генерирует наблюдения за 30 дней до кризиса и 30 дней во время кризиса. Затем создаётся второй набор со сверхдисперсией и различиями между днями недели. Все числовые значения заданы для учебной симуляции; они не являются статистикой ГИБДД.

In [ ]:
# Симулированные данные для домашнего задания
hw_rng = np.random.default_rng(2026)

lambda_before = 8.0        # ожидаемое число ДТП в день до кризиса
exposure_factor = 0.8      # e: трафик во время кризиса относительно «до»
risk_factor = 1.1          # r: риск на единицу трафика относительно «до»
rho_true = exposure_factor * risk_factor
lambda_during = lambda_before * rho_true

days_before = days_during = 30
y_before = hw_rng.poisson(lam=lambda_before, size=days_before)
y_during = hw_rng.poisson(lam=lambda_during, size=days_during)

print(f"Заданное в симуляции отношение интенсивностей rho = {rho_true:.3f}  "
      f"(lambda_before = {lambda_before:.2f} -> lambda_during = {lambda_during:.2f})")
print(f"До:       среднее {y_before.mean():.2f}, SD {y_before.std(ddof=1):.2f}, "
      f"диапазон {y_before.min()}-{y_before.max()}")
print(f"Во время: среднее {y_during.mean():.2f}, SD {y_during.std(ddof=1):.2f}, "
      f"диапазон {y_during.min()}-{y_during.max()}")
print()

# Вторая версия тех же данных: сверхдисперсная, с эффектом дня недели.
# Понадобится в пункте 5.
alpha_disp = 4.0
weekday_mult = np.array([0.9, 0.9, 0.95, 1.0, 1.2, 1.3, 0.9])   # Пн..Вс


def overdispersed(lam_base, n_days, rng_):
    wd = np.arange(n_days) % 7
    lam = lam_base * weekday_mult[wd] * rng_.gamma(alpha_disp, 1 / alpha_disp, n_days)
    return rng_.poisson(lam)


y_before_od = overdispersed(lambda_before, days_before, hw_rng)
y_during_od = overdispersed(lambda_during, days_during, hw_rng)
print(f"Сверхдисперсная версия — во время кризиса: среднее {y_during_od.mean():.2f}, "
      f"дисперсия {y_during_od.var(ddof=1):.2f}, "
      f"индекс {y_during_od.var(ddof=1) / y_during_od.mean():.2f}")

### Задания

1. **Изучите данные и выберите правдоподобие.** Постройте гистограммы `y_before` и `y_during`. Для каждого периода вычислите отношение выборочной дисперсии к среднему, как в примере 2. Насколько результаты согласуются с пуассоновской моделью?
2. **Обоснуйте априорные предположения.** Выберите гамма-приор для интенсивности ДТП в каждом периоде. Поясните, какие значения среднего числа ДТП в день вы считаете правдоподобными. Если используете разные приоры для периодов, обоснуйте различие. Постройте априорные предиктивные распределения и оцените, не придают ли они чрезмерную вероятность значениям, малоправдоподобным для выбранного региона.
3. **Получите апостериорные оценки.** Примените формулу Gamma–Poisson из примера 2 отдельно к каждому периоду. При независимых приорах сгенерируйте независимые апостериорные выборки интенсивностей и вычислите `rho_draws = lam_during_draws / lam_before_draws`. Оцените $\rho$, постройте 95%-й апостериорный интервал и сопоставьте его с заданным в симуляции значением $\rho = 0{,}88$.
4. **Проверьте предсказания.** Для каждого периода проведите апостериорную предиктивную проверку по стандартному отклонению, максимуму и доле нулей. Исходные данные сгенерированы из пуассоновской модели, поэтому систематического несоответствия не ожидается. При этом отдельные расхождения возможны из-за случайной вариативности; интерпретируйте результаты в совокупности.
5. **Повторите оценивание и проверку для `y_before_od` и `y_during_od`.** В этих данных интенсивность меняется между днями, поэтому предположение об общей интенсивности внутри периода нарушено. Покажите, какие статистики обнаруживают несоответствие. Объясните, почему ширину апостериорного интервала для $\rho$ нельзя оценивать без проверки модели.
6. **Подготовьте краткое заключение для комиссии.** Укажите оценённое изменение числа ДТП, неопределённость оценки и ограничения анализа. Обсудите невозможность разделить $e$ и $r$ по этим данным. Поясните также, почему сравнение двух периодов само по себе не доказывает причинный эффект кризиса: например, одновременно мог меняться объём движения из-за окончания сезона отпусков.

В работе оцениваются обоснование модели, корректность предиктивных проверок и соответствие выводов возможностям данных. Одного апостериорного интервала без проверки модели недостаточно.

## Воспроизводимость

Выполните ячейку, чтобы сохранить сведения о версиях библиотек, времени запуска и начальном значении генератора случайных чисел.

In [ ]:
import datetime
import scipy
import plotly
print(f"Время запуска: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python: {sys.version.split()[0]}, NumPy: {np.__version__}, Pandas: {pd.__version__}")
print(f"SciPy: {scipy.__version__}, Plotly: {plotly.__version__}")
print(f"Начальное значение генератора: {RANDOM_SEED}")
print("Данные: учебные извлечения из ROS-Examples (Gelman, Hill, Vehtari),")
print("        загружены из репозитория курса; см. раздел 4.")